In [1]:
# importing libraries

import numpy as np
import pandas as pd
import duckdb
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_pickle('data/intermediate_files/ranker_train_df.pkl')
val_df = pd.read_pickle('data/intermediate_files/ranker_validation_df.pkl')
test_df = pd.read_pickle('data/intermediate_files/ranker_test_df.pkl')

### Preprocessing

##### Deterministic feature creations

In [3]:
def preprocess_pre_sample(df):
    tp_df = df.copy()

    # Time features
    tp_df["query_time"] = pd.to_datetime(tp_df["query_time"])
    tp_df["day"] = tp_df["query_time"].dt.day_name().astype(str)
    tp_df["hour_of_day"] = tp_df["query_time"].dt.hour.astype(str)

    # Log features for skewed numeric columns
    for col in ["past_view_count", "past_atc_count", "past_order_count", "cg_score_raw"]:
        tp_df[col] = tp_df[col].fillna(0)
        tp_df[f"log_{col}"] = np.log1p(tp_df[col].astype(float))

    # Basic null handling
    tp_df["cg_score_norm"] = tp_df["cg_score_norm"].fillna(0)
    tp_df["user_history_bucket"] = tp_df["user_history_bucket"].fillna("unknown").astype(str)

    # Binary columns
    binary_base_cols = [
        "candidate_previously_purchased", "candidate_in_last_k_items",
        "from_user_co_interaction", "from_user_category", "from_user_generality"
    ]
    for col in binary_base_cols:
        if col in tp_df.columns:
            tp_df[col] = tp_df[col].fillna(0).astype("int8")

    # Rank bucket flags; NaN rank becomes 0 for every bucket
    rank_cols = ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    bucket_defs = {
        "1_10": (1, 10),
        "11_50": (11, 50),
        "51_100": (51, 100),
        "101_250": (101, 250),
        "251_500": (251, 500),
    }

    for col in rank_cols:
        if col not in tp_df.columns:
            continue
        for bucket_name, (lo, hi) in bucket_defs.items():
            tp_df[f"{col}_bucket_{bucket_name}"] = tp_df[col].between(lo, hi).fillna(False).astype("int8")

    return tp_df

##### Negative Sampling

In [4]:

def sample_negatives_by_query(df, label_col="label_engagement", query_col="global_query_id", neg_per_pos=10, random_state=42):
    sampled_groups = []
    rng = np.random.default_rng(random_state)

    for _, group in df.groupby(query_col):
        pos = group[group[label_col] == 1]
        neg = group[group[label_col] == 0]

        # Skip all-negative queries for first ranker training
        if len(pos) == 0:
            continue

        n_neg = min(len(neg), len(pos) * neg_per_pos)

        if n_neg > 0:
            neg_sample = neg.sample(n=n_neg, random_state=int(rng.integers(0, 1_000_000)))
            sampled_groups.append(pd.concat([pos, neg_sample], axis=0))
        else:
            sampled_groups.append(pos)

    sampled_df = pd.concat(sampled_groups, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled_df

##### Building the sparse preprocessor

In [5]:
continuous_cols = ["log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_cg_score_raw", "cg_score_norm",]

categorical_cols = ["user_history_bucket", "day", "hour_of_day"]

binary_cols = ["candidate_previously_purchased", "candidate_in_last_k_items", "from_user_co_interaction", "from_user_category",
    "from_user_generality"]

rank_bucket_cols = [
    f"{rank_col}_bucket_{bucket}"
    for rank_col in ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    for bucket in ["1_10", "11_50", "51_100", "101_250", "251_500"]
]

def get_existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def build_sparse_preprocessor(train_feature_df):
    final_continuous = get_existing_cols(train_feature_df, continuous_cols)
    final_categorical = get_existing_cols(train_feature_df, categorical_cols)
    final_binary = get_existing_cols(train_feature_df, binary_cols + rank_bucket_cols)

    feature_cols = final_continuous + final_categorical + final_binary

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), final_continuous),
            ("cat", OneHotEncoder(sparse_output=True, handle_unknown="ignore"), final_categorical),
            ("bin", "passthrough", final_binary),
        ],
        sparse_threshold=1.0
    )

    return preprocessor, feature_cols, final_continuous, final_categorical, final_binary

### Train Data Creation

In [6]:
def prepare_train_matrix(train_df, label_col="label_engagement", neg_per_pos=10, random_state=42):
    # Deterministic features first
    train_pre = preprocess_pre_sample(train_df)

    # Negative sampling only on train
    train_sampled = sample_negatives_by_query(
        train_pre,
        label_col=label_col,
        query_col="global_query_id",
        neg_per_pos=neg_per_pos,
        random_state=random_state
    )

    # Build sparse preprocessor from sampled train
    preprocessor, feature_cols, cont_cols, cat_cols, bin_cols = build_sparse_preprocessor(train_sampled)

    # Fit sparse transformer
    X_train = preprocessor.fit_transform(train_sampled[feature_cols])
    y_train = train_sampled[label_col].astype(int).values

    return X_train, y_train, train_sampled, preprocessor, feature_cols

In [7]:
X_train, y_train, train_sampled, preprocessor, feature_cols = prepare_train_matrix(
    train_df,
    label_col="label_engagement",
    neg_per_pos=10,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("y_train mean:", y_train.mean())
print("train_sampled shape:", train_sampled.shape)

X_train shape: (60577, 66)
y_train mean: 0.09090909090909091
train_sampled shape: (60577, 55)


### Model Training

In [8]:
lr_model_trial = LogisticRegression(solver='saga', penalty='l2', n_jobs=-1, verbose=1, C=1.0, max_iter=300)

In [9]:
lr_model_trial.fit(X_train, y_train)

convergence after 28 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

### Validation/test Data prep

In [10]:
def align_feature_cols(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    return df[feature_cols]

def predict_scores_in_chunks(df, preprocessor, model, feature_cols, chunk_size=300_000):
    scores = np.empty(len(df), dtype=np.float32)

    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))

        chunk = df.iloc[start:end].copy()
        chunk = preprocess_pre_sample(chunk)
        chunk_X = preprocessor.transform(align_feature_cols(chunk, feature_cols))

        scores[start:end] = model.predict_proba(chunk_X)[:, 1].astype(np.float32)
        print(f"Scored rows {start:,} to {end:,}")

    return scores

In [11]:
# sampling validation/test to make it smaller, might comment this later

def sample_query_groups(df, n_queries, query_col='global_query_id', random_state=42):

    query_ids = df[query_col].drop_duplicates()
    sample_ids = query_ids.sample(n=min(n_queries, len(query_ids)), random_state=random_state)

    return df[df[query_col].isin(sample_ids)].copy()

In [12]:
# validation data
val_small = sample_query_groups(val_df, 10000, query_col='global_query_id', random_state=42)

val_scored = val_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
val_scored["lr_score"] = predict_scores_in_chunks(
    val_small, preprocessor, lr_model_trial, feature_cols, chunk_size=300_000
)
val_scored["cg_baseline_score"] = -val_scored["cg_rank"]

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,010,291


### Evaluation - Ranker

In [13]:
def evaluate_ranking(df, score_col, label_col="label_engagement", query_col="global_query_id",
                     ks=[10, 20, 50], mode="reranker_only"):
    eval_df = df.copy()

    if mode == "reranker_only":
        
        q_pos = eval_df.groupby(query_col)[label_col].sum()
        keep_qids = q_pos[q_pos > 0].index
        eval_df = eval_df[eval_df[query_col].isin(keep_qids)].copy()

    eval_df = eval_df.sort_values([query_col, score_col, "cg_rank"], ascending=[True, False, True])

    rows = []
    n_queries = eval_df[query_col].nunique()

    for k in ks:
        hit_list, recall_list, ndcg_list, mrr_list = [], [], [], []

        for _, g in eval_df.groupby(query_col, sort=False):
            labels = g[label_col].to_numpy()
            total_pos = labels.sum()

            if total_pos == 0:
                hit_list.append(0.0)
                recall_list.append(0.0)
                ndcg_list.append(0.0)
                mrr_list.append(0.0)
                continue

            topk = labels[:k]
            hits = topk.sum()

            hit = 1.0 if hits > 0 else 0.0
            recall = hits / total_pos

            discounts = 1.0 / np.log2(np.arange(2, len(topk) + 2))
            dcg = np.sum(topk * discounts)

            ideal_hits = int(min(total_pos, k))
            ideal_discounts = 1.0 / np.log2(np.arange(2, ideal_hits + 2))
            idcg = np.sum(ideal_discounts)
            ndcg = dcg / idcg if idcg > 0 else 0.0

            pos_idx = np.where(topk == 1)[0]
            mrr = 1.0 / (pos_idx[0] + 1) if len(pos_idx) > 0 else 0.0

            hit_list.append(hit)
            recall_list.append(recall)
            ndcg_list.append(ndcg)
            mrr_list.append(mrr)

        rows.append({
            "score_col": score_col,
            "mode": mode,
            "K": k,
            "n_queries": n_queries,
            "HitRate": np.mean(hit_list),
            "Recall": np.mean(recall_list),
            "NDCG": np.mean(ndcg_list),
            "MRR": np.mean(mrr_list),
        })

    return pd.DataFrame(rows)

#### Lift scores

In [14]:
def add_lift(compare_df, baseline_score_col="cg_baseline_score", model_score_col="lr_score"):
    base = compare_df[compare_df["score_col"] == baseline_score_col].copy()
    model = compare_df[compare_df["score_col"] == model_score_col].copy()

    merged = model.merge(
        base,
        on=["mode", "K"],
        suffixes=("_model", "_baseline")
    )

    for metric in ["HitRate", "Recall", "NDCG", "MRR"]:
        merged[f"{metric}_lift_abs"] = merged[f"{metric}_model"] - merged[f"{metric}_baseline"]
        merged[f"{metric}_lift_pct"] = np.where(
            merged[f"{metric}_baseline"] > 0,
            100 * merged[f"{metric}_lift_abs"] / merged[f"{metric}_baseline"],
            np.nan
        )

    return merged

##### Compare CG baseline vs Ranker - Positive queries  

In [15]:
ks = [10, 20, 50, 100]

val_cg_rerank = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

val_lr_rerank = evaluate_ranking(
    val_scored,
    score_col="lr_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

val_compare_rerank = pd.concat([val_cg_rerank, val_lr_rerank], ignore_index=True)
val_compare_rerank

,score_col,mode,K,n_queries,HitRate,Recall,NDCG,MRR
0,cg_baseline_score,reranker_only,10,197,0.314721,0.278765,0.163966,0.144130
1,cg_baseline_score,reranker_only,20,197,0.436548,0.395939,0.194895,0.152713
2,cg_baseline_score,reranker_only,50,197,0.659898,0.619036,0.241261,0.159662
3,cg_baseline_score,reranker_only,100,197,0.847716,0.818359,0.275667,0.162436
4,lr_score,reranker_only,10,197,0.695431,0.677157,0.561387,0.528215
5,lr_score,reranker_only,20,197,0.736041,0.716920,0.572648,0.531306
6,lr_score,reranker_only,50,197,0.817259,0.792978,0.588266,0.533765
7,lr_score,reranker_only,100,197,0.873096,0.858968,0.599995,0.534528


##### Compare CG baseline vs Ranker - All queries  

In [16]:
val_cg_all = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

val_lr_all = evaluate_ranking(
    val_scored,
    score_col="lr_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

val_compare_all = pd.concat([val_cg_all, val_lr_all], ignore_index=True)
val_compare_all

,score_col,mode,K,n_queries,HitRate,Recall,NDCG,MRR
0,cg_baseline_score,all_queries,10,10000,0.0062,0.005492,0.003230,0.002839
1,cg_baseline_score,all_queries,20,10000,0.0086,0.007800,0.003839,0.003008
2,cg_baseline_score,all_queries,50,10000,0.0130,0.012195,0.004753,0.003145
3,cg_baseline_score,all_queries,100,10000,0.0167,0.016122,0.005431,0.003200
4,lr_score,all_queries,10,10000,0.0137,0.013340,0.011059,0.010406
5,lr_score,all_queries,20,10000,0.0145,0.014123,0.011281,0.010467
6,lr_score,all_queries,50,10000,0.0161,0.015622,0.011589,0.010515
7,lr_score,all_queries,100,10000,0.0172,0.016922,0.011820,0.010530


In [17]:
val_lift_rerank = add_lift(val_compare_rerank)
val_lift_all = add_lift(val_compare_all)

val_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.163966,0.561387,0.397421,242.379626,0.314721,0.695431,0.380711,120.967742
1,reranker_only,20,0.194895,0.572648,0.377754,193.824585,0.436548,0.736041,0.299492,68.604651
2,reranker_only,50,0.241261,0.588266,0.347005,143.829934,0.659898,0.817259,0.157360,23.846154
3,reranker_only,100,0.275667,0.599995,0.324328,117.651972,0.847716,0.873096,0.025381,2.994012


##### Hyperparameter tuning

In [18]:
def tune_logistic_C(X_train, y_train, val_df, preproc, feature_cols,
                    C_grid, ks=[10, 20], label_col="label_engagement"):
    rows = []
    models = {}

    for C in C_grid:
        print(f"\nTraining C={C}")

        model = LogisticRegression(
            solver="saga",
            penalty="l2",
            C=C,
            max_iter=500,
            n_jobs=-1,
            verbose=0,
            random_state=42
        )

        model.fit(X_train, y_train)

        val_scored = val_df[["global_query_id", "candidate_itemid", "cg_rank", label_col]].copy()
        
        val_scored["lr_score"] = predict_scores_in_chunks(
            val_df,
            preprocessor=preproc,
            model=model,
            feature_cols = feature_cols,
            chunk_size=300_000
        )
        val_scored["cg_baseline_score"] = -val_scored["cg_rank"]

        metrics = evaluate_ranking(
            val_scored,
            score_col="lr_score",
            label_col=label_col,
            ks=ks,
            mode="reranker_only"
        )

        metrics["C"] = C
        models[C] = model
        rows.append(metrics)

    return pd.concat(rows, ignore_index=True), models

In [19]:
# tuning the regularization parameter - lambda

C_grid = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

tuning_results, models = tune_logistic_C(
    X_train=X_train,
    y_train=y_train,
    val_df=val_small,
    preproc=preprocessor,
    feature_cols=feature_cols,
    C_grid=C_grid,
    ks=[10, 20],
    label_col="label_engagement"
)

tuning_results.sort_values(["K", "NDCG"], ascending=[True, False])


Training C=0.01
Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,010,291

Training C=0.03
Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,010,291

Training C=0.1
Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100

,score_col,mode,K,n_queries,HitRate,Recall,NDCG,MRR,C
6,lr_score,reranker_only,10,197,0.695431,0.677157,0.561912,0.528892,0.30
8,lr_score,reranker_only,10,197,0.695431,0.677157,0.561387,0.528215,1.00
10,lr_score,reranker_only,10,197,0.690355,0.672081,0.559920,0.527707,3.00
12,lr_score,reranker_only,10,197,0.690355,0.672081,0.559697,0.527453,10.00
4,lr_score,reranker_only,10,197,0.690355,0.672081,0.559257,0.526861,0.10
2,lr_score,reranker_only,10,197,0.690355,0.672081,0.555157,0.523272,0.03
0,lr_score,reranker_only,10,197,0.695431,0.670812,0.554554,0.525931,0.01
7,lr_score,reranker_only,20,197,0.736041,0.716920,0.573158,0.531959,0.30
5,lr_score,reranker_only,20,197,0.741117,0.721997,0.572948,0.530496,0.10
9,lr_score,reranker_only,20,197,0.736041,0.716920,0.572648,0.531306,1.00


### Test Scores

In [20]:
best_C = 0.3 # selecting best C basis NDCG@20

test_small = sample_query_groups(test_df, 10000, query_col='global_query_id', random_state=42)

test_scored = test_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
test_scored["lr_score"] = predict_scores_in_chunks(
    test_small, preprocessor, models[best_C], feature_cols, chunk_size=300_000
)
test_scored["cg_baseline_score"] = -test_scored["cg_rank"]

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,065,555


##### Positive Queries

In [21]:
ks = [10, 20, 50, 100]

test_cg_rerank = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

test_lr_rerank = evaluate_ranking(
    test_scored,
    score_col="lr_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

test_compare_rerank = pd.concat([test_cg_rerank, test_lr_rerank], ignore_index=True)
test_compare_rerank

,score_col,mode,K,n_queries,HitRate,Recall,NDCG,MRR
0,cg_baseline_score,reranker_only,10,177,0.276836,0.249718,0.156688,0.137075
1,cg_baseline_score,reranker_only,20,177,0.389831,0.353296,0.184515,0.145030
2,cg_baseline_score,reranker_only,50,177,0.593220,0.553296,0.226430,0.151635
3,cg_baseline_score,reranker_only,100,177,0.836158,0.803296,0.268702,0.155234
4,lr_score,reranker_only,10,177,0.711864,0.675706,0.513115,0.476863
5,lr_score,reranker_only,20,177,0.779661,0.755461,0.534835,0.481474
6,lr_score,reranker_only,50,177,0.853107,0.839266,0.552436,0.483963
7,lr_score,reranker_only,100,177,0.920904,0.904237,0.563392,0.484922


##### All Queries

In [22]:
test_cg_all = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

test_lr_all = evaluate_ranking(
    test_scored,
    score_col="lr_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

test_compare_all = pd.concat([test_cg_all, test_lr_all], ignore_index=True)
test_compare_all

,score_col,mode,K,n_queries,HitRate,Recall,NDCG,MRR
0,cg_baseline_score,all_queries,10,10000,0.0049,0.004420,0.002773,0.002426
1,cg_baseline_score,all_queries,20,10000,0.0069,0.006253,0.003266,0.002567
2,cg_baseline_score,all_queries,50,10000,0.0105,0.009793,0.004008,0.002684
3,cg_baseline_score,all_queries,100,10000,0.0148,0.014218,0.004756,0.002748
4,lr_score,all_queries,10,10000,0.0126,0.011960,0.009082,0.008440
5,lr_score,all_queries,20,10000,0.0138,0.013372,0.009467,0.008522
6,lr_score,all_queries,50,10000,0.0151,0.014855,0.009778,0.008566
7,lr_score,all_queries,100,10000,0.0163,0.016005,0.009972,0.008583


In [ ]:
test_lift_rerank = add_lift(test_compare_rerank)
test_lift_all = add_lift(test_compare_all)

test_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.156688,0.513115,0.356427,227.476408,0.276836,0.711864,0.435028,157.142857
1,reranker_only,20,0.184515,0.534835,0.350319,189.859001,0.389831,0.779661,0.389831,100.000000
2,reranker_only,50,0.226430,0.552436,0.326006,143.976470,0.593220,0.853107,0.259887,43.809524
3,reranker_only,100,0.268702,0.563392,0.294691,109.672072,0.836158,0.920904,0.084746,10.135135


### Export Data

In [25]:
import joblib

# dumping the model 
joblib.dump(models[best_C], "models/wide_model/logistic_wide_reranker_c0_3.pkl")
joblib.dump(preprocessor, "models/wide_model/logistic_wide_preprocessor.pkl")
joblib.dump(feature_cols, "models/wide_model/logistic_wide_feature_cols.pkl")

# exporting test results
test_compare_rerank.to_csv("reports/wide_model/test_reranker_only_metrics.csv", index=False)
test_compare_all.to_csv("reports/wide_model/test_all_query_metrics.csv", index=False)